In [ ]:
import os
import warnings
from copy import deepcopy
from itertools import combinations, cycle
from typing import Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import scanpy as sc
import scirpy as ir
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from anndata import AnnData
from matplotlib import cm
from matplotlib.cm import get_cmap
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, PathPatch
from matplotlib.path import Path
from scipy.spatial import distance as sc_distance
from scipy.stats import mannwhitneyu, ttest_ind

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42
warnings.filterwarnings("ignore")

os.makedirs("figures", exist_ok=True)


In [ ]:
CD8_T = sc.read_h5ad( 'CD8_T.h5ad')


In [ ]:
tcr=sc.read_h5ad('tcr.h5ad')
ir.pp.merge_with_ir(CD8_T,tcr)
ir.tl.chain_qc(CD8_T)


In [ ]:
CD8_T_clone=CD8_T[CD8_T.obs["receptor_subtype"].isin(["TRA+TRB"])].copy()
CD8_T_clone=CD8_T_clone[CD8_T_clone.obs["chain_pairing"].isin(["single pair","extra VJ","extra VDJ","two full chains"])].copy()


In [ ]:
ir.pp.ir_dist(CD8_T_clone)
ir.tl.define_clonotypes(CD8_T_clone, receptor_arms="all", dual_ir="primary_only", within_group="Sample_ID",key_added='clone_Sample')
ir.tl.define_clonotypes(CD8_T_clone, receptor_arms="all", dual_ir="primary_only", within_group="Patient_ID",key_added='clone_Patient')
CD8_T_clone.write_h5ad('CD8_T_clone.h5ad', compression='gzip')


In [ ]:
cd8_color_map = {
    "CD8_TN_c2":    "#9ccc65",
    "CD8_TN_c9":    "#689f38",
    "CD8_t-TEM_c8": "#ffecb3",
    "CD8_TEM_c1":   "#ffd54f",
    "CD8_TEM_c3":   "#ffc107",
    "CD8_TEM_c5":   "#ffa000",
    "CD8_TEM_c6":   "#ff6f00",
    "CD8_TRM_c4":   "#8d6e63",
    "CD8_TRM_c11":  "#5d4037",
    "CD8_TISG_c13": "#0097a7",
    "CD8_TEMRA_c7": "#ef5350",
    "CD8_TNK_c10":  "#e53935",
    "CD8_TEX_c12":  "#9c27b0"
}


In [ ]:
meta_color_map = {
    "Child": "#fdae6b",
    "Adult": "#9ecae1",
    "NB": "#e6550d",
    "HB":  "#fd8d3c",
    "WT":  "#ffed6f",
    'PB': "#ffffb3" ,
    "MB":  "#e7969c",
    'pLGG':  "#d6616b",
    'pHGG':  "#ad494a",
    "BRCA":"#3182bd",
    'CRC':"#6baed6",
    'PC':"#8ca252",
    "LC":"#b5cf6b",
    "HCC":"#756BB1",
    'RC':"#bebada",
    'aHCC':"#a55194",
}


In [ ]:
disease_order = ['pLGG','pHGG','MB','NB','WT','HB','PB',
                'aHCC','BRCA','CRC','HCC','LC','RC','PC']


# Fig2a

In [ ]:
anno_order = ["CD8_TN_c2","CD8_TN_c9",
              'CD8_t-TEM_c8',
              "CD8_TEM_c1","CD8_TEM_c3","CD8_TEM_c5","CD8_TEM_c6",
              'CD8_TRM_c4',"CD8_TRM_c11",
              'CD8_TEMRA_c7','CD8_TNK_c10','CD8_TEX_c12',
              'CD8_TISG_c13']

CD8_T.obs["anno_CD8T"] = CD8_T.obs["anno_CD8T"].astype("category")
CD8_T.obs["anno_CD8T"] = CD8_T.obs["anno_CD8T"].cat.set_categories(anno_order, ordered=True)


In [ ]:
ax = sc.pl.umap(
    CD8_T,
    color="T_sub_level_3",
    palette=cd8_color_map,
    use_raw=False,
    frameon=False,
    size=3,
    legend_loc="right margin",
    show=False,
)

for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig('figures/Fig2a_CD8_umap.pdf', dpi=600, bbox_inches="tight")


# Fig2b

In [ ]:
base_cmap = cm.get_cmap("coolwarm")

trunc_cmap = LinearSegmentedColormap.from_list(
    "trunc_coolwarm",
    base_cmap(np.linspace(0.10, 0.90, 256))
)


In [ ]:
marker_genes_dict = {
    "Naive": ["TCF7","LEF1"],
    "Central": ["CCR7","SELL"],
    "Activation": ["CD69","HLA-DRB1"],
    "Resident": ["ITGAE","ITGA1","CXCR6"],
    "Cytolytic": ["GZMK","GZMB","PRF1","GNLY"],
    "Terminal":["ZEB2","TBX21"],
    "NK-like":["KIR2DL4","KLRD1"],
    "Exhausted": ["TOX","PDCD1"],
    "ISG":["ISG15","IFITM1"],
}


In [ ]:
available_genes = set(CD8_T.var_names)

filtered_marker_genes_dict = {
    category: [gene for gene in genes if gene in available_genes]
    for category, genes in marker_genes_dict.items()
}

filtered_marker_genes_dict = {
    k: v for k, v in filtered_marker_genes_dict.items() if v
}

if not filtered_marker_genes_dict:
    raise ValueError("没有找到可以用于绘图的基因，请检查 marker_genes_dict 和数据集中的基因名是否匹配。")

ax=sc.pl.dotplot(
    CD8_T,
    filtered_marker_genes_dict,
    'T_sub_level_3',
    dendrogram=False,
    cmap=trunc_cmap,
    standard_scale="var",
    colorbar_title="column scaled\nexpression",
    use_raw=False,
    return_fig=True,
    show=False
)

ax.savefig('figures/Fig2b_CD8_markers.pdf', bbox_inches="tight")
ax.show()


# FigS2a

In [ ]:
score= pd.read_csv('CD8_score.csv')
score_dict = {col: score[col].dropna().tolist() for col in score.columns}
signature_names = list(score_dict.keys())
for signature_name, gene_list in score_dict.items():
    sc.tl.score_genes(CD8_T, gene_list=gene_list, score_name=signature_name,use_raw=False)


In [ ]:
matrixplot=sc.pl.matrixplot(
    CD8_T,
    signature_names,
     'T_sub_level_3',
    dendrogram=False,
    cmap=trunc_cmap,
    standard_scale="var",
    colorbar_title="column scaled\nscore",
    return_fig=True
)

matrixplot.savefig('figures/FigS2a_CD8_scores.pdf', bbox_inches="tight")
matrixplot.show()


# Fig2c

In [ ]:
for sig in signature_names:
    col = CD8_T.obs[sig]
    col_min = col.min()
    col_max = col.max()
    denom = col_max - col_min

    if denom != 0:
        CD8_T.obs[f"{sig}_norm"] = (col - col_min) / denom
    else:

        CD8_T.obs[f"{sig}_norm"] = 0


In [ ]:
CD8_T_tumor= CD8_T[CD8_T.obs['Sample_Type'].isin(['T'])].copy()


In [ ]:
axes = sc.pl.umap(
    CD8_T,
    color=["Naive_norm", "Activation:Effector function_norm", "Cytotoxicity_norm", "Exhaustion_norm"],
    cmap=trunc_cmap,
    size=4,
    use_raw=False,
    show=False
)

if not isinstance(axes, (list, np.ndarray)):
    axes = [axes]

for ax in np.ravel(axes):
    for coll in ax.collections:
        coll.set_rasterized(True)

plt.savefig('figures/Fig2c_CD8_score_umap.pdf',dpi=600, bbox_inches="tight")
plt.show()


# Fig2d

In [ ]:
def _fdr_bh(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    mask = np.isfinite(p)
    if mask.sum() == 0:
        return q
    p_nonan = p[mask]
    m = p_nonan.size
    order = np.argsort(p_nonan)
    ranked = p_nonan[order]
    q_ordered = ranked * m / (np.arange(1, m + 1))
    q_ordered = np.minimum.accumulate(q_ordered[::-1])[::-1]
    inv = np.empty_like(order)
    inv[order] = np.arange(m)
    q_nonan = np.minimum(q_ordered[inv], 1.0)
    q[mask] = q_nonan
    return q

def plot_composed_composition_and_table(
    obs_df: pd.DataFrame,
    *,
    sample_id_col: str = "Sample_ID",
    group_col: str = "Period_merged",
    celltype_col: str = "annotation_TT",
    group_levels: Tuple[str, str] = ("Adult", "Child"),
    celltype_order: Optional[List[str]] = None,
    palette: Optional[Dict[str, str]] = None,
    figsize: Tuple[float, float] = (7.5, 6.0),
    bar_width: float = 0.7,
    alpha_sig: float = 0.05,
    fdr_correction: bool = False,
    cmap_name: str = "coolwarm",
    diff_bar_width: float = 0.4,
    title: Optional[str] = None,
    save: Optional[str] = None,

    effect: str = "diff",
    effect_eps: float = 1e-9,
    color_grey_for_nonsig: str = "#D3D3D3",

    dedup_celltype: str = "first",

    stack_alpha: float = 0.95,
    flow_alpha: float = 0.40,
    diff_alpha: float = 1.00,
):
    A, B = group_levels
    eff_key = effect.lower()
    if eff_key not in {"diff", "snd", "lrr", "hedgesg"}:
        raise ValueError("effect must be one of: 'diff', 'snd', 'lrr', 'hedgesg'.")
    if dedup_celltype not in {"first", "mean"}:
        raise ValueError("dedup_celltype must be 'first' or 'mean'.")

    df = obs_df.loc[
        obs_df[group_col].isin([A, B]),
        [sample_id_col, group_col, celltype_col]
    ].copy()
    if df.empty:
        raise ValueError("筛选后数据为空；请检查 group_levels / group_col / 数据内容。")

    group_counts = df.groupby([group_col, celltype_col]).size().unstack(fill_value=0)
    idx = [g for g in [A, B] if g in group_counts.index]
    group_counts = group_counts.reindex(idx)

    if celltype_order is not None:
        cols = [c for c in celltype_order if c in group_counts.columns]
        group_counts = group_counts[cols]

    cell_types = group_counts.columns.tolist()
    group_props = group_counts.div(group_counts.sum(axis=1), axis=0).fillna(0.0)

    if palette is None:
        raw_colors = (
            list(cm.Greys(np.linspace(0.2, 0.4, 3))) +
            list(cm.Greens(np.linspace(0.2, 0.6, 4))) +
            list(cm.Oranges(np.linspace(0.2, 0.4, 2))) +
            list(cm.Reds(np.linspace(0.4, 0.6, 3))) +
            list(cm.Purples(np.linspace(0.4, 0.6, 1))) +
            list(cm.Blues(np.linspace(0.4, 0.4, 1))) +
            list(cm.YlOrBr(np.linspace(0.2, 0.6, 1)))
        )

        def to_morandi(c, mix=0.8, grey=(0.88, 0.86, 0.84)):
            r, g, b, a = c
            r_new = r * mix + grey[0] * (1 - mix)
            g_new = g * mix + grey[1] * (1 - mix)
            b_new = b * mix + grey[2] * (1 - mix)
            return (r_new, g_new, b_new, a)

        color_pool = [to_hex(to_morandi(c)) for c in raw_colors]
        if len(cell_types) > len(color_pool):
            colors = [next(cycle(color_pool)) for _ in cell_types]
        else:
            colors = color_pool[:len(cell_types)]
        palette = dict(zip(cell_types, colors))

    color_map = {ct: palette.get(ct, "#808080") for ct in cell_types}

    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 2, width_ratios=[2.0, 0.7], wspace=0.35, figure=fig)
    axL = fig.add_subplot(gs[0, 0])
    axR = fig.add_subplot(gs[0, 1])

    x_locs = np.arange(len(group_props.index))
    bottoms = np.zeros(len(group_props), dtype=float)

    for ct in cell_types:
        vals = group_props[ct].values
        axL.bar(
            x_locs, vals, bottom=bottoms, width=bar_width,
            color=color_map[ct], label=ct, linewidth=0,
            alpha=stack_alpha
        )
        bottoms += vals

    for ct in cell_types:
        col = color_map[ct]
        pos_ct = cell_types.index(ct)
        for i in range(len(group_props.index) - 1):
            cur = group_props.iloc[i]
            nxt = group_props.iloc[i + 1]
            v0 = float(cur.get(ct, 0.0))
            v1 = float(nxt.get(ct, 0.0))
            if v0 == 0 and v1 == 0:
                continue
            y0_base = float(cur.iloc[:pos_ct].sum()) if pos_ct > 0 else 0.0
            y1_base = float(nxt.iloc[:pos_ct].sum()) if pos_ct > 0 else 0.0
            verts = [
                (x_locs[i] + bar_width / 2, y0_base),
                (x_locs[i + 1] - bar_width / 2, y1_base),
                (x_locs[i + 1] - bar_width / 2, y1_base + v1),
                (x_locs[i] + bar_width / 2, y0_base + v0),
            ]
            axL.add_patch(
                patches.Polygon(
                    verts,
                    closed=True,
                    facecolor=col,
                    edgecolor="none",
                    alpha=flow_alpha
                )
            )

    axL.set_xticks(x_locs)
    axL.set_xticklabels(group_props.index)
    axL.set_ylabel("Proportion")
    axL.set_xlim(-0.5, len(x_locs) - 0.5)
    axL.set_ylim(0, 1)
    axL.margins(x=0.02)
    axL.spines[["top", "right"]].set_visible(False)

    diff_top2bottom = list(reversed(cell_types))

    counts = df.groupby([sample_id_col, celltype_col]).size().unstack(fill_value=0)
    props = counts.div(counts.sum(axis=1), axis=0).fillna(0.0)
    info = df[[sample_id_col, group_col]].drop_duplicates()
    props = props.merge(info, left_index=True, right_on=sample_id_col, how="left")

    long_df = pd.melt(
        props,
        id_vars=[sample_id_col, group_col],
        var_name=celltype_col,
        value_name="Proportion"
    )

    rows = []
    for ct, sub in long_df.groupby(celltype_col, sort=False):
        a = sub.loc[sub[group_col] == A, "Proportion"].astype(float)
        b = sub.loc[sub[group_col] == B, "Proportion"].astype(float)

        mean_a = float(a.mean()) if len(a) else np.nan
        mean_b = float(b.mean()) if len(b) else np.nan
        diff = mean_b - mean_a

        eff = np.nan
        if np.isfinite(mean_a) and np.isfinite(mean_b):
            if eff_key == "diff":
                eff = diff
            elif eff_key == "snd":
                eff = diff / (mean_a + mean_b + float(effect_eps))
            elif eff_key == "lrr":
                eff = float(np.log((mean_b + float(effect_eps)) / (mean_a + float(effect_eps))))
            elif eff_key == "hedgesg":
                if len(a) >= 2 and len(b) >= 2:
                    sa = float(a.std(ddof=1))
                    sb = float(b.std(ddof=1))
                    na, nb = int(len(a)), int(len(b))
                    sp2 = ((na - 1) * sa * sa + (nb - 1) * sb * sb) / max(na + nb - 2, 1)
                    sp = float(np.sqrt(sp2)) if sp2 > 0 else np.nan
                    if np.isfinite(sp) and sp > 0:
                        d = diff / sp
                        J = 1.0 - (3.0 / (4.0 * (na + nb) - 9.0)) if (na + nb) > 2 else 1.0
                        eff = float(J * d)

        p = np.nan
        try:
            if len(a) and len(b):
                p = mannwhitneyu(a, b, alternative="two-sided")[1]
        except Exception:
            p = np.nan

        rows.append((ct, mean_a, mean_b, diff, eff, p, int(len(a)), int(len(b))))

    summary_df = pd.DataFrame(
        rows,
        columns=[celltype_col, "Mean_" + A, "Mean_" + B, "Diff", "Effect", "P_value", "n_" + A, "n_" + B]
    ).set_index(celltype_col)

    if fdr_correction:
        summary_df["Q_value"] = _fdr_bh(summary_df["P_value"].values)
        stat_col = "Q_value"
    else:
        stat_col = "P_value"

    summary_df = summary_df.reindex([ct for ct in diff_top2bottom if ct in summary_df.index])

    if summary_df.index.duplicated().any():
        if dedup_celltype == "first":
            summary_df = summary_df[~summary_df.index.duplicated(keep="first")]
        else:
            mean_cols = ["Mean_" + A, "Mean_" + B, "Diff", "Effect", "P_value"]
            if "Q_value" in summary_df.columns:
                mean_cols.append("Q_value")
            sum_cols = ["n_" + A, "n_" + B]
            agg = {c: "mean" for c in mean_cols if c in summary_df.columns}
            agg.update({c: "sum" for c in sum_cols if c in summary_df.columns})
            summary_df = summary_df.groupby(level=0).agg(agg)

    existing = summary_df.index.tolist()

    diff_prop_series = group_props.loc[B, existing].fillna(0.0)
    total = float(diff_prop_series.sum()) or 1.0
    heights = (diff_prop_series / total).values

    cmap = get_cmap(cmap_name)
    eff_vals = summary_df["Effect"].values.astype(float)
    vmax = float(np.nanmax(np.abs(eff_vals))) if np.isfinite(eff_vals).any() else 1.0
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    norm = Normalize(vmin=-vmax, vmax=vmax)

    diff_bar_width = float(diff_bar_width)
    if diff_bar_width <= 0:
        diff_bar_width = 0.01
    if diff_bar_width > 1:
        diff_bar_width = 1.0

    axR.set_xlim(0, 1)
    axR.set_ylim(0, 1)
    axR.spines[["top", "right", "left", "bottom"]].set_visible(False)
    axR.set_xticks([0.5])

    eff_label = {"diff": "Diff", "snd": "SND", "lrr": "LRR", "hedgesg": "Hedges_g"}[eff_key]
    axR.set_xticklabels([eff_label], fontsize=11)
    axR.set_yticks([])

    x0 = 0.5 - diff_bar_width / 2.0
    w = diff_bar_width

    eps_draw = 1e-6
    cum = 0.0
    for i, ct in enumerate(existing):
        h = float(heights[i])
        if h <= 0:
            continue
        y = 1.0 - (cum + h) - eps_draw
        h_draw = h + 2 * eps_draw

        statv = summary_df.at[ct, stat_col]
        sig = pd.notna(statv) and np.isfinite(float(statv)) and (float(statv) <= float(alpha_sig))

        effv = summary_df.at[ct, "Effect"]
        effv = float(effv) if (pd.notna(effv) and np.isfinite(float(effv))) else np.nan

        face = cmap(norm(effv)) if (sig and np.isfinite(effv)) else color_grey_for_nonsig

        rect = plt.Rectangle(
            (x0, max(0.0, y)),
            w,
            min(1.0 - max(0.0, y), h_draw),
            facecolor=face,
            edgecolor="none",
            linewidth=0,
            alpha=diff_alpha
        )
        axR.add_patch(rect)
        cum += h

    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, ax=axR, fraction=0.06, pad=0.04)

    if eff_key == "diff":
        cbar_label = f"{B} - {A}"
    elif eff_key == "snd":
        cbar_label = f"SND: ({B}-{A})/({A}+{B})"
    elif eff_key == "lrr":
        cbar_label = f"LRR: log(({B}+eps)/({A}+eps))"
    else:
        cbar_label = f"Hedges' g ({B}-{A})"

    cbar.set_label(cbar_label, fontsize=10)
    cbar.outline.set_visible(False)
    ticks = np.linspace(-vmax, vmax, 5)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f"{t:.3f}" for t in ticks])

    legend_order = list(reversed(cell_types))
    handles = [mpl.patches.Patch(facecolor=color_map[ct], label=ct) for ct in legend_order]
    fig.legend(
        handles, legend_order,
        bbox_to_anchor=(1.1, 0.5),
        loc="center left",
        ncol=1,
        fontsize="x-small",
        frameon=False,
        title="Cell_Type"
    )

    if title is None:
        title = f"Composition (Left) and {eff_label} Bar (Right): {A} → {B}"
    fig.suptitle(title, y=0.995, fontsize=13)

    plt.tight_layout(rect=(0, 0.02, 0.9, 0.98))

    if save:
        fig.savefig(save, bbox_inches="tight")

    return fig, (axL, axR), summary_df


In [ ]:
obs_df= CD8_T.obs[CD8_T.obs["Sample_Type"] == "T"].copy()

fig, (axL, axR), summary = plot_composed_composition_and_table(
    obs_df=obs_df,
    sample_id_col="Sample_ID",
    group_col="Period_merged",
    celltype_col="anno_CD8T",
    group_levels=("Adult","Child"),
    effect = "snd",

    palette=cd8_color_map,
    figsize=(6,7),
    alpha_sig=0.05,
    fdr_correction=True,
    cmap_name='coolwarm',
    bar_width=0.6,
    diff_bar_width=0.3,
    stack_alpha=0.85,
    flow_alpha=0.25,
    diff_alpha=1.00,
    save='figures/Fig2d_CD8_composition.pdf'
)


# Fig2f

In [ ]:
def plot_and_test_scores_by_sample(
    adata,

    cell_filter_col=None,
    cell_filter_values=None,
    cell_filter_query=None,

    sample_col="Sample_ID",
    group_cols=("Disease", "Period_merged", "Period"),
    score_cols=("Naive", "Activation:Effector function", "Cytotoxicity", "Exhaustion"),
    how_group="mode",
    dropna_sample=True,
    add_diff_pair=None,
    min_cells_per_sample: Optional[int] = None,

    group_col="Period_merged",
    value_col="Exhaustion_mean",
    order=None,
    figsize: Tuple[float, float] = (6, 4),

    palette: Optional[Dict[str, str]] = None,
    subgroup_col: Optional[str] = None,
    subgroup_order: Optional[Sequence[str]] = None,
    subgroup_palette: Optional[Dict[str, str]] = None,
    meta_color_map: Optional[Dict[str, str]] = None,

    jitter: float = 0.08,
    point_size: float = 20,
    point_alpha: float = 0.9,
    min_box_height_ratio: float = 0.03,
    random_state: int = 0,

    show_subgroup_legend: bool = True,
    box_edge_color: str = "#C0C0C0",
    box_line_alpha: float = 0.6,
    box_line_width: float = 1.2,
    box_width: float = 0.30,

    ylabel: Optional[str] = None,
    xlabel: Optional[str] = None,
    title: Optional[str] = None,
    x_rotation: float = 45,

    size_col=None,
    size_sqrt: bool = True,
    S_MIN: float = 10,
    S_MAX: float = 90,

    test_method="mannwhitney",

    return_data=True,
    show=True,
):

    rng = np.random.default_rng(random_state)

    def summarize_scores_by_sample(
        adata_in,
        sample_col=sample_col,
        group_cols=None,
        score_cols=score_cols,
        how_group=how_group,
        dropna_sample=dropna_sample,
        add_diff_pair=add_diff_pair,
    ):
        if group_cols is None:
            group_cols = []

        adata_in.obs.columns = adata_in.obs.columns.str.strip()

        want_cols = [sample_col] + list(group_cols) + list(score_cols)
        missing = [c for c in want_cols if c not in adata_in.obs.columns]
        if missing:
            raise KeyError(f"Missing in adata.obs: {missing}")

        df0 = adata_in.obs[want_cols].copy()
        if dropna_sample:
            df0 = df0.dropna(subset=[sample_col])

        def pick(x):
            if how_group == "first":
                return x.iloc[0]
            m = x.mode()
            return m.iat[0] if len(m) else x.iloc[0]

        agg = {"n_cells": (score_cols[0], "size")}
        for sc in score_cols:
            agg[f"{sc}_mean"] = (sc, "mean")
        for gc in group_cols:
            agg[gc] = (gc, pick)

        out = df0.groupby(sample_col).agg(**agg).reset_index()

        if add_diff_pair is not None:
            a, b = add_diff_pair
            if (a in score_cols) and (b in score_cols):
                out[f"{a}_minus_{b}"] = out[f"{a}_mean"] - out[f"{b}_mean"]

        return out

    def bh_fdr(pvals):
        pvals = np.asarray(pvals, dtype=float)
        out = np.full_like(pvals, np.nan, dtype=float)
        ok = ~np.isnan(pvals)
        idx = np.where(ok)[0]
        if len(idx) == 0:
            return out
        pv = pvals[idx]
        order_idx = np.argsort(pv)
        pv_sorted = pv[order_idx]
        ranks = np.arange(1, len(pv_sorted) + 1)
        q = pv_sorted * len(pv_sorted) / ranks
        q = np.minimum.accumulate(q[::-1])[::-1]
        q = np.clip(q, 0, 1)
        out[idx[order_idx]] = q
        return out

    def pairwise_tests(df, group_col, value_col, groups=None, method="mannwhitney"):
        d = df[[group_col, value_col]].dropna()
        if groups is None:
            groups = d[group_col].astype(str).unique().tolist()

        rows = []
        for g1, g2 in combinations(groups, 2):
            x = d.loc[d[group_col] == g1, value_col].to_numpy(dtype=float)
            y = d.loc[d[group_col] == g2, value_col].to_numpy(dtype=float)
            n1, n2 = len(x), len(y)

            if n1 < 2 or n2 < 2:
                p = np.nan
                test = method
            else:
                if method == "ttest":
                    _, p = ttest_ind(x, y, equal_var=False, nan_policy="omit")
                    test = "Welch t-test"
                else:
                    _, p = mannwhitneyu(x, y, alternative="two-sided")
                    test = "Mann–Whitney U"

            rows.append({"group1": g1, "group2": g2, "n1": n1, "n2": n2, "test": test, "p": p})

        out = pd.DataFrame(rows)
        out["p_adj"] = bh_fdr(out["p"].values)
        out = out.sort_values(["p_adj", "p"], na_position="last").reset_index(drop=True)
        return out

    def make_auto_palette(levels, used_colors=None):
        base_tab20 = list(cm.get_cmap("tab20").colors)

        def to_morandi(c, mix=0.85, grey=(0.94, 0.94, 0.94)):
            r, g, b = c[:3]
            return (
                r * mix + grey[0] * (1 - mix),
                g * mix + grey[1] * (1 - mix),
                b * mix + grey[2] * (1 - mix),
            )

        auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if used_colors is not None:
            used_colors = set(c.lower() for c in used_colors)
            auto_colors = [c for c in auto_colors if c.lower() not in used_colors]

        if len(auto_colors) == 0:
            auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if len(auto_colors) < len(levels):
            repeats = int(np.ceil(len(levels) / len(auto_colors)))
            auto_colors = (auto_colors * repeats)[:len(levels)]
        else:
            auto_colors = auto_colors[:len(levels)]

        return dict(zip(levels, auto_colors))

    adata_sub = adata

    if cell_filter_query is not None:
        mask = adata_sub.obs.eval(cell_filter_query)
        adata_sub = adata_sub[mask].copy()
    elif (cell_filter_col is not None) and (cell_filter_values is not None):
        adata_sub = adata_sub[adata_sub.obs[cell_filter_col].isin(list(cell_filter_values))].copy()

    df_sample = summarize_scores_by_sample(
        adata_sub,
        sample_col=sample_col,
        group_cols=list(group_cols),
        score_cols=tuple(score_cols),
        how_group=how_group,
        dropna_sample=dropna_sample,
        add_diff_pair=add_diff_pair,
    )

    if min_cells_per_sample is not None:
        df_sample = df_sample.loc[df_sample["n_cells"] >= min_cells_per_sample].copy()

    if df_sample.empty:
        raise ValueError("按当前筛选条件和 min_cells_per_sample 过滤后，没有剩余样本可用于作图。")

    if value_col not in df_sample.columns:
        raise KeyError(f"value_col='{value_col}' not found in df_sample columns: {df_sample.columns.tolist()}")

    if group_col not in df_sample.columns:
        raise KeyError(f"group_col='{group_col}' not found in df_sample columns: {df_sample.columns.tolist()}")

    use_cols = [group_col, value_col]
    if subgroup_col is not None:
        if subgroup_col not in df_sample.columns:
            raise KeyError(f"subgroup_col='{subgroup_col}' not found in df_sample columns: {df_sample.columns.tolist()}")
        use_cols.append(subgroup_col)
    if size_col is not None:
        if size_col not in df_sample.columns:
            raise KeyError(f"size_col='{size_col}' not found in df_sample columns: {df_sample.columns.tolist()}")
        use_cols.append(size_col)

    sub = df_sample[use_cols].dropna(subset=[group_col, value_col]).copy()
    if sub.empty:
        raise ValueError("筛选后 df_sample 为空，请检查 group_col / value_col / subgroup_col。")

    if order is None:
        group_order = list(pd.unique(sub[group_col].dropna()))
    else:
        group_order = [g for g in order if g in sub[group_col].values]

    if len(group_order) == 0:
        raise ValueError("order 中没有有效分组。")

    all_vals = sub[value_col].values.astype(float)
    y_min, y_max = float(np.min(all_vals)), float(np.max(all_vals))
    if y_min == y_max:
        y_min -= 0.5
        y_max += 0.5
    y_range = y_max - y_min
    min_box_h = y_range * min_box_height_ratio

    if palette is None:
        palette = {}
        if meta_color_map is not None:
            for g in group_order:
                if g in meta_color_map:
                    palette[g] = meta_color_map[g]
        for g in group_order:
            if g not in palette:
                palette[g] = box_edge_color

    if subgroup_col is not None:
        if subgroup_order is None:
            sub_levels = list(pd.unique(sub[subgroup_col].dropna()))
        else:
            sub_levels = [s for s in subgroup_order if s in sub[subgroup_col].values]

        final_subgroup_palette = {}

        if meta_color_map is not None:
            for s in sub_levels:
                if s in meta_color_map:
                    final_subgroup_palette[s] = meta_color_map[s]

        if subgroup_palette is not None:
            for s in sub_levels:
                if s in subgroup_palette:
                    final_subgroup_palette[s] = subgroup_palette[s]

        missing_levels = [s for s in sub_levels if s not in final_subgroup_palette]
        if len(missing_levels) > 0:
            auto_pal = make_auto_palette(missing_levels, used_colors=final_subgroup_palette.values())
            final_subgroup_palette.update(auto_pal)

        subgroup_palette = final_subgroup_palette
    else:
        subgroup_palette = {}
        sub_levels = []

    if size_col is not None and size_col in sub.columns:
        s_raw = sub[size_col].to_numpy(dtype=float)
        s_raw = np.sqrt(s_raw) if size_sqrt else s_raw
        g_smin, g_smax = float(np.min(s_raw)), float(np.max(s_raw))
    else:
        g_smin, g_smax = 0.0, 1.0

    fig, ax = plt.subplots(figsize=figsize)
    xs = np.arange(len(group_order)) * 0.5

    whisker_cap_width = box_width / 4

    for x, g in zip(xs, group_order):
        g_mask = sub[group_col] == g
        vals = sub.loc[g_mask, value_col].values.astype(float)
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue

        if len(vals) >= 2:
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
        else:
            med = vals[0]
            q1 = med - min_box_h / 2
            q3 = med + min_box_h / 2

        if (q3 - q1) < min_box_h:
            center = (q1 + q3) / 2
            q1 = center - min_box_h / 2
            q3 = center + min_box_h / 2

        w_low = float(np.min(vals))
        w_high = float(np.max(vals))

        box_col = palette.get(g, box_edge_color)

        rect = plt.Rectangle(
            (x - box_width / 2, q1),
            box_width,
            q3 - q1,
            facecolor="none",
            edgecolor=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=1,
        )
        ax.add_patch(rect)

        ax.plot(
            [x - box_width / 2, x + box_width / 2],
            [med, med],
            color=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=2,
        )

        ax.plot([x, x], [q3, w_high], color=box_col, linewidth=box_line_width, alpha=box_line_alpha, zorder=1)
        ax.plot([x, x], [q1, w_low], color=box_col, linewidth=box_line_width, alpha=box_line_alpha, zorder=1)
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_high, w_high],
            color=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=1,
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_low, w_low],
            color=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=1,
        )

        if subgroup_col is None:
            y = vals
            jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(y))

            if size_col is not None and size_col in sub.columns:
                s0 = sub.loc[g_mask, size_col].to_numpy(dtype=float)
                s0 = s0[np.isfinite(sub.loc[g_mask, value_col].to_numpy(dtype=float))]
                s0 = np.sqrt(s0) if size_sqrt else s0
                if abs(g_smax - g_smin) < 1e-12:
                    sizes = np.full(len(y), (S_MIN + S_MAX) / 2.0, dtype=float)
                else:
                    sizes = S_MIN + (s0 - g_smin) / (g_smax - g_smin) * (S_MAX - S_MIN)
            else:
                sizes = np.full(len(y), point_size, dtype=float)

            ax.scatter(
                np.full_like(y, x, dtype=float) + jitter_x,
                y,
                s=sizes,
                color=box_col,
                alpha=point_alpha,
                edgecolors="none",
                zorder=3,
            )
        else:
            extra_cols = [size_col] if (size_col is not None and size_col in sub.columns) else []
            g_df = sub.loc[g_mask, [value_col, subgroup_col] + extra_cols]

            for sub_name, sdf in g_df.groupby(subgroup_col):
                v = sdf[value_col].values.astype(float)
                v = v[np.isfinite(v)]
                if len(v) == 0:
                    continue

                jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(v))
                col = subgroup_palette.get(sub_name, "#555555")

                if size_col is not None and size_col in sdf.columns:
                    s0 = sdf[size_col].values.astype(float)
                    s0 = s0[np.isfinite(sdf[value_col].values.astype(float))]
                    s0 = np.sqrt(s0) if size_sqrt else s0
                    if abs(g_smax - g_smin) < 1e-12:
                        sizes = np.full(len(v), (S_MIN + S_MAX) / 2.0, dtype=float)
                    else:
                        sizes = S_MIN + (s0 - g_smin) / (g_smax - g_smin) * (S_MAX - S_MIN)
                else:
                    sizes = np.full(len(v), point_size, dtype=float)

                ax.scatter(
                    np.full_like(v, x, dtype=float) + jitter_x,
                    v,
                    s=sizes,
                    color=col,
                    alpha=point_alpha,
                    edgecolors="none",
                    zorder=3,
                    label=sub_name,
                )

    ax.set_xticks(xs)
    ax.set_xticklabels(group_order, rotation=x_rotation, ha="right")
    ax.set_ylim(y_min, y_max)
    ax.margins(x=0.05)

    ax.set_ylabel(ylabel if ylabel is not None else value_col)
    ax.set_xlabel("" if xlabel is None else xlabel)
    if title is not None:
        ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if subgroup_col is not None and show_subgroup_legend:
        handles, labels = ax.get_legend_handles_labels()
        handle_by_label = dict(zip(labels, handles))

        if subgroup_order is not None:
            legend_labels = [s for s in subgroup_order if s in handle_by_label]
        else:
            legend_labels = [s for s in sub_levels if s in handle_by_label]

        legend_handles = [handle_by_label[l] for l in legend_labels]

        ax.legend(
            legend_handles,
            legend_labels,
            title=subgroup_col,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
            fontsize="x-small"
        )

    pval_table = pairwise_tests(df_sample, group_col, value_col, groups=group_order, method=test_method)

    if show:
        plt.show()

    if not return_data:
        return fig, ax

    return {
        "adata_sub": adata_sub,
        "df_sample": df_sample,
        "pval_table": pval_table,
        "fig": fig,
        "ax": ax,
        "order": group_order,
    }


In [ ]:
res = plot_and_test_scores_by_sample(
    CD8_T_tumor,
    score_cols=("Exhaustion","Cytotoxicity"),
    cell_filter_col="anno_CD8T",
    cell_filter_values=["CD8_TEX_c12"],
    group_cols=("Disease", "Period_merged", "Period"),
    group_col="Period_merged",
    value_col="Exhaustion_mean",
    order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    meta_color_map=meta_color_map,
    figsize=(2, 3.5),
    box_line_alpha=0.3,
    box_line_width=1.2,
    box_width=0.30,
    point_size=20,
    point_alpha=1.0,
    show=False,
    min_cells_per_sample=5,

)

plt.savefig('figures/Fig2f_TEX_exhaustion.pdf', dpi=300, bbox_inches="tight")
plt.show
df_sample = res["df_sample"]
pval_table = res["pval_table"]
df_sample.head()
pval_table


# FigS2G

In [ ]:
res = plot_and_test_scores_by_sample(
    CD8_T_tumor,
    score_cols=("Exhaustion","Cytotoxicity"),
    cell_filter_col="anno_CD8T",
    cell_filter_values=["CD8_TEX_c12"],
    group_cols=("Disease", "Period_merged", "Period"),
    group_col="Period_merged",
    value_col="Cytotoxicity_mean",
    order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    meta_color_map=meta_color_map,
    figsize=(2, 3.5),
    box_line_alpha=0.3,
    box_line_width=1.2,
    box_width=0.30,
    point_size=20,
    point_alpha=1.0,
    show=False,
    min_cells_per_sample=5,

)

plt.savefig('figures/FigS2g_TEX_cytotoxicity.pdf', dpi=300, bbox_inches="tight")
plt.show
df_sample = res["df_sample"]
pval_table = res["pval_table"]
df_sample.head()
pval_table


In [ ]:
res = plot_and_test_scores_by_sample(
    CD8_T_tumor,
    score_cols=("Exhaustion","Cytotoxicity"),
    cell_filter_col="anno_CD8T",
    cell_filter_values=["CD8_TNK_c10","CD8_TEMRA_c7"],
    group_cols=("Disease", "Period_merged", "Period"),
    group_col="Period_merged",
    value_col="Cytotoxicity_mean",
    order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    meta_color_map=meta_color_map,
    figsize=(2, 3.5),
    box_line_alpha=0.3,
    box_line_width=1.2,
    box_width=0.30,
    point_size=20,
    point_alpha=1.0,
    show=False,
    min_cells_per_sample=5,

)

plt.savefig('figures/FigS2g_TNK_TEMRA_cytotoxicity.pdf', dpi=300, bbox_inches="tight")
plt.show
df_sample = res["df_sample"]
pval_table = res["pval_table"]
df_sample.head()
pval_table


# FigS2b

In [ ]:
CD8_T.obs["Origin_Period_Sample"] = (
    CD8_T.obs["Origin"].map(lambda x: "inhouse" if x == "Dong_lab" else "published").astype(str)
    + "_"
    + CD8_T.obs["Period_merged"].astype(str)
    + "_"
    + CD8_T.obs["Sample_Type"].astype(str)
)
print(CD8_T.obs['Origin_Period_Sample'].value_counts())


In [ ]:
outdir = 'figures'
os.makedirs(outdir, exist_ok=True)

viridis = mpl.cm.get_cmap("viridis")

umap_df = pd.DataFrame(CD8_T.obsm["X_umap"], columns=["UMAP1", "UMAP2"])
umap_df["Origin_Period_Sample"] = CD8_T.obs["Origin_Period_Sample"].values

subset_all = umap_df

groups = [
    "published_Adult_T",
    "published_Child_T",
    "inhouse_Child_B",
    "inhouse_Child_T"
]

kws_background = dict(
    fill=True,
    cmap=mpl.colors.ListedColormap([viridis(0.0)]),
    bw_adjust=0.7,
    levels=50,
    thresh=0.01
)

kws_foreground = dict(
    fill=True,
    cmap="viridis",
    bw_adjust=0.5,
    levels=50,
    thresh=0.025
)

x_min, x_max = subset_all["UMAP1"].min(), subset_all["UMAP1"].max()
y_min, y_max = subset_all["UMAP2"].min(), subset_all["UMAP2"].max()
x_margin = (x_max - x_min) * 0.05
y_margin = (y_max - y_min) * 0.05

for g in groups:
    subset_fg = umap_df[umap_df["Origin_Period_Sample"] == g]

    plt.figure(figsize=(7, 6))

    sns.kdeplot(
        data=subset_all,
        x="UMAP1",
        y="UMAP2",
        **kws_background
    )

    sns.kdeplot(
        data=subset_fg,
        x="UMAP1",
        y="UMAP2",
        **kws_foreground
    )

    plt.xlim(x_min - x_margin, x_max + x_margin)
    plt.ylim(y_min - y_margin, y_max + y_margin)

    plt.gca().set_axis_off()
    plt.tight_layout()

    outfile = os.path.join(outdir, f"FigS2b_{g}_density.tiff")
    plt.savefig(outfile, format="tiff", dpi=300, bbox_inches="tight", transparent=True)
    plt.show()
    plt.close()


# FigS2c

In [ ]:
def compute_celltype_fraction_per_sample(
    obs_df: pd.DataFrame,
    *,
    sample_id_col: str = "Sample_ID",
    disease_col: str = "Disease",
    celltype_col: str = "anno_TKA"
):

    sample_disease = (
        obs_df[[sample_id_col, disease_col]]
        .dropna(subset=[sample_id_col, disease_col])
        .drop_duplicates(subset=[sample_id_col])
    )

    counts = (
        obs_df
        .groupby([sample_id_col, celltype_col])
        .size()
        .reset_index(name="n")
    )

    totals = (
        counts
        .groupby(sample_id_col)["n"]
        .sum()
        .reset_index(name="total")
    )

    frac_df = counts.merge(totals, on=sample_id_col, how="left")
    frac_df["frac"] = frac_df["n"] / frac_df["total"]

    frac_df = frac_df.merge(sample_disease, on=sample_id_col, how="left")

    frac_df = frac_df[
        [sample_id_col, disease_col, celltype_col, "n", "total", "frac"]
    ]

    count_matrix = (
        counts.pivot(index=sample_id_col,
                     columns=celltype_col,
                     values="n")
        .fillna(0)
        .astype(int)
    )

    count_matrix["total"] = totals.set_index(sample_id_col)["total"]

    count_matrix = count_matrix.join(
        sample_disease.set_index(sample_id_col),
        how="left"
    )

    return frac_df, count_matrix


In [ ]:
def plot_box_by_disease(
    frac_df: pd.DataFrame,
    *,
    disease_col: str = "Disease",
    sample_id_col: str = "Sample_ID",
    celltype_col: str = "annotation_TT",
    celltypes: List[str] = None,
    disease_order: List[str] = None,
    figsize: Tuple[float, float] = None,
    show_pct: bool = True,
    random_state: int = 0,
    min_box_height_ratio: float = 0.03,

    palette: Optional[Dict[str, str]] = None,
    meta_color_map: Optional[Dict[str, str]] = None,

    all_celltypes_for_palette: Optional[List[str]] = None,
    obs_celltype_series: Optional[pd.Series] = None,
    pair_tests: Optional[List[Tuple[str, str]]] = None,
    test_method: str = "mannwhitney",

    sig_line_y_ratio: float = 0.05,
    sig_text_offset_ratio: float = 0.01,
    sig_line_pad: float = 0.05,
):
    rng = np.random.default_rng(random_state)

    if celltypes is None:
        celltypes = frac_df[celltype_col].unique().tolist()

    if disease_order is None:
        default_order = [
            "MB", "pLGG", "pHGG", 'aHCC', "WT",
            'RC', "HB", "HCC", 'PB', 'PC',
            "NB", "BRCA", "CRC", "LC"
        ]
        present = frac_df[disease_col].dropna().unique().tolist()
        disease_order = [d for d in default_order if d in present]

    n_ct = len(celltypes)
    if figsize is None:
        figsize = (max(8, len(disease_order) * 0.7), 2.3 * n_ct)

    if palette is None:
        if all_celltypes_for_palette is not None:
            all_ct = [ct for ct in all_celltypes_for_palette
                      if ct in frac_df[celltype_col].unique()]
        elif obs_celltype_series is not None:
            s = obs_celltype_series
            if hasattr(s, "cat"):
                try:
                    cats = list(s.cat.categories)
                    all_ct = [ct for ct in cats if ct in frac_df[celltype_col].unique()]
                except Exception:
                    all_ct = list(pd.unique(s))
            else:
                all_ct = list(pd.unique(s))
                all_ct = [ct for ct in all_ct if ct in frac_df[celltype_col].unique()]
        else:
            all_ct = frac_df[celltype_col].unique().tolist()

        palette = {}

        if meta_color_map is not None:
            for ct in all_ct:
                if ct in meta_color_map:
                    palette[ct] = meta_color_map[ct]

        missing_ct = [ct for ct in all_ct if ct not in palette]
        if len(missing_ct) > 0:
            raw_colors = (
                list(cm.Greys(np.linspace(0.2, 0.4, 2))) +
                list(cm.Greens(np.linspace(0.2, 0.6, 7))) +
                list(cm.Oranges(np.linspace(0.2, 0.4, 7))) +
                list(cm.Reds(np.linspace(0.4, 0.6, 4))) +
                list(cm.Purples(np.linspace(0.4, 0.6, 1))) +
                list(cm.Blues(np.linspace(0.4, 0.4, 1))) +
                list(cm.YlOrBr(np.linspace(0.2, 0.6, 1)))
            )

            def to_morandi(c, mix=0.8, grey=(0.88, 0.86, 0.84)):
                r, g, b, a = c
                return (
                    r * mix + grey[0] * (1 - mix),
                    g * mix + grey[1] * (1 - mix),
                    b * mix + grey[2] * (1 - mix),
                    a
                )

            color_pool = [to_hex(to_morandi(c)) for c in raw_colors]
            used_colors = set(c.lower() for c in palette.values())
            color_pool = [c for c in color_pool if c.lower() not in used_colors]

            if len(color_pool) == 0:
                color_pool = [to_hex(to_morandi(c)) for c in raw_colors]

            if len(missing_ct) > len(color_pool):
                fill_colors = [next(cycle(color_pool)) for _ in missing_ct]
            else:
                fill_colors = color_pool[:len(missing_ct)]

            for ct, c in zip(missing_ct, fill_colors):
                palette[ct] = c

    ct_color = {ct: palette.get(ct, "#808080") for ct in celltypes}

    if pair_tests is None:
        pair_tests = [
            ("pHGG", 'aHCC'),
            ("WT", 'RC'),
            ("HB", "HCC"),
            ('PB', 'PC'),
        ]

    fig, axes = plt.subplots(n_ct, 1, figsize=figsize, sharex=False)
    if n_ct == 1:
        axes = [axes]

    xs = np.arange(len(disease_order)) * 0.6

    for ax, ct in zip(axes, celltypes):
        sub = frac_df.loc[frac_df[celltype_col] == ct].copy()
        if sub.empty:
            ax.text(0.5, 0.5, f"No data for {ct}", ha="center", va="center")
            ax.axis("off")
            continue

        if show_pct:
            sub["value"] = sub["frac"] * 100
        else:
            sub["value"] = sub["frac"]

        all_vals = sub["value"].values
        if len(all_vals) == 0:
            y_min, y_max = 0.0, 1.0
        else:
            y_min, y_max = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))
            if y_min == y_max:
                y_min -= 0.5
                y_max += 0.5

        y_range = y_max - y_min
        min_box_h = y_range * min_box_height_ratio if y_range > 0 else 0.1

        box_data = [
            sub.loc[sub[disease_col] == d, "value"].values
            for d in disease_order
        ]

        col = ct_color.get(ct, "#808080")
        box_width = 0.5

        for x, vals in zip(xs, box_data):
            if len(vals) == 0:
                continue

            vals = np.asarray(vals, dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                continue

            if len(vals) >= 2:
                q1, med, q3 = np.percentile(vals, [25, 50, 75])
            else:
                med = vals[0]
                q1 = med - min_box_h / 2
                q3 = med + min_box_h / 2

            if (q3 - q1) < min_box_h:
                center = (q1 + q3) / 2
                q1 = center - min_box_h / 2
                q3 = center + min_box_h / 2

            w_low = float(np.min(vals))
            w_high = float(np.max(vals))

            rect = plt.Rectangle(
                (x - box_width / 2, q1),
                box_width,
                q3 - q1,
                facecolor="none",
                edgecolor=col,
                linewidth=1.0,
                zorder=1
            )
            ax.add_patch(rect)

            ax.plot(
                [x - box_width / 2, x + box_width / 2],
                [med, med],
                color=col, linewidth=1.2, zorder=2
            )

            ax.plot([x, x], [q3, w_high], color=col, linewidth=1.0, zorder=1)
            ax.plot([x, x], [q1, w_low], color=col, linewidth=1.0, zorder=1)
            ax.plot(
                [x - box_width / 4, x + box_width / 4],
                [w_high, w_high],
                color=col, linewidth=1.0, zorder=1
            )
            ax.plot(
                [x - box_width / 4, x + box_width / 4],
                [w_low, w_low],
                color=col, linewidth=1.0, zorder=1
            )

        for x, d in zip(xs, disease_order):
            vals = sub.loc[sub[disease_col] == d, "value"].values
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                continue
            jitter = rng.normal(loc=0.0, scale=0.08, size=len(vals))
            ax.scatter(
                x + jitter,
                vals,
                s=14,
                alpha=0.85,
                color=col,
                edgecolors="none",
                zorder=3
            )

        max_annot_y = y_max
        if test_method.lower() != "none" and y_range > 0:
            line_y = y_max + sig_line_y_ratio * y_range
            text_y = line_y + sig_text_offset_ratio * y_range

            for (d1, d2) in pair_tests:
                if (d1 not in disease_order) or (d2 not in disease_order):
                    continue

                v1 = sub.loc[sub[disease_col] == d1, "value"].values
                v2 = sub.loc[sub[disease_col] == d2, "value"].values
                v1 = v1[np.isfinite(v1)]
                v2 = v2[np.isfinite(v2)]

                if len(v1) < 2 or len(v2) < 2:
                    continue

                if test_method.lower() == "mannwhitney":
                    p = mannwhitneyu(v1, v2, alternative="two-sided").pvalue
                elif test_method.lower() == "ttest":
                    p = ttest_ind(v1, v2, equal_var=False).pvalue
                else:
                    continue

                if p <= 0.0001:
                    label = "****"
                elif p <= 0.001:
                    label = "***"
                elif p <= 0.01:
                    label = "**"
                elif p <= 0.05:
                    label = "*"
                else:
                    label = "ns"

                i1 = disease_order.index(d1)
                i2 = disease_order.index(d2)
                x1, x2 = xs[i1], xs[i2]
                x_left, x_right = min(x1, x2), max(x1, x2)

                ax.plot(
                    [x_left + sig_line_pad, x_right - sig_line_pad],
                    [line_y, line_y],
                    color="black",
                    linewidth=0.8,
                    zorder=4
                )

                ax.text(
                    (x_left + x_right) / 2.0,
                    text_y,
                    label,
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    zorder=5
                )

                max_annot_y = max(max_annot_y, text_y + 0.03 * y_range)

        ax.set_ylim(y_min, max_annot_y)

        ax.set_ylabel("Proportion (%)")
        ax.set_title(ct, fontsize=11)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for ax in axes:
        ax.set_xticks(xs)
        ax.set_xticklabels(disease_order, rotation=45, ha="right")

    plt.tight_layout(h_pad=1.0)
    return fig, axes


In [ ]:
frac_df, count_matrix = compute_celltype_fraction_per_sample(
    CD8_T_tumor.obs,
    sample_id_col="Sample_ID",
    disease_col="Disease",
    celltype_col="anno_CD8T"
)


In [ ]:
celltypes_to_plot = ["CD8_TISG_c13","CD8_TEX_c12","CD8_TRM_c4","CD8_TEM_c5","CD8_TEM_c3","CD8_t-TEM_c8"]
my_order = ["NB","MB",'pLGG','pHGG','aHCC',"WT",
            'RC',"HB","HCC",'PB','PC'
            ,"BRCA","CRC","LC"]
pairs = [
    ('pHGG', 'aHCC'),
    ("WT",   'RC'),
    ("HB",   "HCC"),
    ('PB',  'PC'),
]

fig, axes = plot_box_by_disease(
    frac_df,
    disease_col="Disease",
    sample_id_col="Sample_ID",
    celltype_col="anno_CD8T",
    celltypes=celltypes_to_plot,
    disease_order=my_order,
    pair_tests=pairs,

    obs_celltype_series=CD8_T.obs["anno_CD8T"],
    figsize=(6, 8),
    show_pct=True,
    meta_color_map=cd8_color_map
)

fig.savefig('figures/FigS2c_CD8_composition_by_tumor.pdf', format="pdf", bbox_inches="tight")
plt.show()


# clone分析

# Fig2e

In [ ]:
CD8_T_clone= sc.read_h5ad('CD8_T_clone.h5ad')


In [ ]:
CD8_T_clone_tumor= CD8_T_clone[CD8_T_clone.obs['Sample_Type'].isin(['T'])].copy()


In [ ]:
def scirpy_expansion_index(
    adata,
    groupby=("patient", "cluster"),
    target_col="clone_id",
    dropna_clonotype=True,
    log_base="e",
    return_details=False,
    neg_epsilon=1e-12,
    min_cells_per_group=None,
    min_clonotypes_per_group=None,
):

    if isinstance(groupby, str):
        group_cols = [groupby]
    else:
        group_cols = list(groupby)

    obs = adata.obs.copy()

    if dropna_clonotype:
        obs = obs[~obs[target_col].isna()].copy()

    cell_counts = (
        obs.groupby(group_cols + [target_col])
           .size()
           .rename("n_cells")
           .reset_index()
    )

    if cell_counts.empty:
        cols = group_cols + [
            "expansion_index",
            "pielou_evenness",
            "shannon",
            "n_clonotypes",
            "n_cells",
        ]
        if return_details:
            return (
                pd.DataFrame(columns=cols),
                pd.DataFrame(columns=group_cols + [target_col, "n_cells", "p"]),
            )
        else:
            return pd.DataFrame(columns=cols)

    cell_counts["group_total"] = cell_counts.groupby(group_cols)["n_cells"].transform("sum")
    cell_counts["p"] = cell_counts["n_cells"] / cell_counts["group_total"]

    if log_base == "e":
        _log = np.log
    elif log_base == 2:
        _log = lambda x: np.log(x) / np.log(2)
    elif log_base == 10:
        _log = lambda x: np.log(x) / np.log(10)
    else:
        raise ValueError("log_base 只能是 'e'、2 或 10")

    rows = []
    for keys, df_g in cell_counts.groupby(group_cols, sort=False):
        keys = (keys,) if not isinstance(keys, tuple) else keys

        p = df_g["p"].to_numpy(dtype=float)
        p = p[p > 0]
        S = int(p.size)

        if S <= 1:
            H = 0.0
            J = np.nan
            EI = np.nan
        else:
            H = float(-(p * _log(p)).sum())
            denom = _log(S)
            J = float(H / denom) if denom > 0 else np.nan
            EI = 1.0 - J

            if EI is not np.nan and EI < 0 and abs(EI) < neg_epsilon:
                EI = 0.0

        n_cells_total = int(df_g["group_total"].iloc[0])
        rows.append((*keys, EI, J, H, S, n_cells_total))

    df_summary = pd.DataFrame(
        rows,
        columns=group_cols
        + ["expansion_index", "pielou_evenness", "shannon", "n_clonotypes", "n_cells"],
    )

    mask = pd.Series(True, index=df_summary.index)

    if min_cells_per_group is not None:
        mask = mask & (df_summary["n_cells"] >= min_cells_per_group)

    if min_clonotypes_per_group is not None:
        mask = mask & (df_summary["n_clonotypes"] >= min_clonotypes_per_group)

    df_summary = df_summary.loc[mask].copy()

    if return_details:
        details = cell_counts[group_cols + [target_col, "n_cells", "p"]].copy()
        if not df_summary.empty:

            valid_groups = df_summary[group_cols].drop_duplicates()
            details = details.merge(valid_groups, on=group_cols, how="inner")
        else:
            details = details.iloc[0:0].copy()
        return df_summary, details

    return df_summary


In [ ]:
df_pc = scirpy_expansion_index(
    CD8_T_clone_tumor,
    groupby=("Sample_ID"),
    target_col="clone_Sample",
    log_base="e",
    min_cells_per_group=20,
    min_clonotypes_per_group=3,
    return_details=False
)


In [ ]:
all_samples = CD8_T_clone_tumor.obs["Sample_ID"].unique()
df_pc = (
    df_pc.set_index("Sample_ID")
         .reindex(all_samples)
         .reset_index()
)


In [ ]:
extra_cols = [ "Disease","Period_merged","Sample_Type"]
agg_fun = lambda s: s.mode().iat[0] if not s.mode().empty else (s.dropna().iloc[0] if s.dropna().size else np.nan)

meta = (CD8_T_clone.obs.groupby("Sample_ID")[extra_cols]
          .agg(agg_fun)
          .reset_index())

df_pc_with_meta = df_pc.merge(meta, on="Sample_ID", how="left")


In [ ]:
def plot_group_box_dots(
    df: pd.DataFrame,
    value_col: str,
    group_col: str,
    *,
    group_order: Optional[Sequence[str]] = None,
    palette: Optional[Dict[str, str]] = None,
    subgroup_col: Optional[str] = None,
    subgroup_order: Optional[Sequence[str]] = None,
    subgroup_palette: Optional[Dict[str, str]] = None,
    meta_color_map: Optional[Dict[str, str]] = None,
    figsize: Tuple[float, float] = (6, 4),
    jitter: float = 0.08,
    point_size: float = 20,
    min_box_height_ratio: float = 0.03,
    random_state: int = 0,
    ylabel: Optional[str] = None,
    xlabel: Optional[str] = None,
    title: Optional[str] = None,
    show_subgroup_legend: bool = True,
    box_edge_color: str = "#C0C0C0",
    box_line_alpha: float = 0.6,
    box_line_width: float = 1.2,
):
    rng = np.random.default_rng(random_state)

    if group_order is None:
        group_order = list(pd.unique(df[group_col].dropna()))
    else:
        group_order = [g for g in group_order if g in df[group_col].values]

    use_cols = [group_col, value_col]
    if subgroup_col is not None:
        use_cols.append(subgroup_col)

    sub = df[use_cols].dropna(subset=[group_col, value_col])
    if sub.empty:
        raise ValueError("筛选后数据为空，请检查 value_col / group_col / subgroup_col。")

    all_vals = sub[value_col].values.astype(float)
    y_min, y_max = float(np.min(all_vals)), float(np.max(all_vals))
    if y_min == y_max:
        y_min -= 0.5
        y_max += 0.5
    y_range = y_max - y_min
    min_box_h = y_range * min_box_height_ratio

    def make_auto_palette(levels, used_colors=None):
        base_tab20 = list(cm.get_cmap("tab20").colors)

        def to_morandi(c, mix=0.85, grey=(0.94, 0.94, 0.94)):
            r, g, b = c[:3]
            return (
                r * mix + grey[0] * (1 - mix),
                g * mix + grey[1] * (1 - mix),
                b * mix + grey[2] * (1 - mix),
            )

        auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if used_colors is not None:
            used_colors = set(c.lower() for c in used_colors)
            auto_colors = [c for c in auto_colors if c.lower() not in used_colors]

        if len(auto_colors) == 0:
            auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if len(auto_colors) < len(levels):
            repeats = int(np.ceil(len(levels) / len(auto_colors)))
            auto_colors = (auto_colors * repeats)[:len(levels)]
        else:
            auto_colors = auto_colors[:len(levels)]

        return dict(zip(levels, auto_colors))

    if palette is None:
        palette = {}
        if meta_color_map is not None:
            for g in group_order:
                if g in meta_color_map:
                    palette[g] = meta_color_map[g]
        for g in group_order:
            if g not in palette:
                palette[g] = box_edge_color

    if subgroup_col is not None:
        if subgroup_order is None:
            sub_levels = list(pd.unique(sub[subgroup_col].dropna()))
        else:
            sub_levels = [s for s in subgroup_order if s in sub[subgroup_col].values]

        final_subgroup_palette = {}

        if meta_color_map is not None:
            for s in sub_levels:
                if s in meta_color_map:
                    final_subgroup_palette[s] = meta_color_map[s]

        if subgroup_palette is not None:
            for s in sub_levels:
                if s in subgroup_palette:
                    final_subgroup_palette[s] = subgroup_palette[s]

        missing_levels = [s for s in sub_levels if s not in final_subgroup_palette]
        if len(missing_levels) > 0:
            auto_pal = make_auto_palette(
                missing_levels,
                used_colors=final_subgroup_palette.values()
            )
            final_subgroup_palette.update(auto_pal)

        subgroup_palette = final_subgroup_palette
    else:
        subgroup_palette = {}
        sub_levels = []

    fig, ax = plt.subplots(figsize=figsize)
    xs = np.arange(len(group_order)) * 0.5
    box_width = 0.30

    whisker_cap_width = box_width / 4
    median_width = box_line_width
    whisker_width = box_line_width
    cap_width = box_line_width
    rect_line_width = box_line_width

    for x, g in zip(xs, group_order):
        g_mask = sub[group_col] == g
        vals = sub.loc[g_mask, value_col].values.astype(float)
        if len(vals) == 0:
            continue

        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue

        if len(vals) >= 2:
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
        else:
            med = vals[0]
            q1 = med - min_box_h / 2
            q3 = med + min_box_h / 2

        if (q3 - q1) < min_box_h:
            center = (q1 + q3) / 2
            q1 = center - min_box_h / 2
            q3 = center + min_box_h / 2

        w_low = float(np.min(vals))
        w_high = float(np.max(vals))

        box_col = palette.get(g, box_edge_color)

        rect = plt.Rectangle(
            (x - box_width / 2, q1),
            box_width,
            q3 - q1,
            facecolor="none",
            edgecolor=box_col,
            linewidth=rect_line_width,
            alpha=box_line_alpha,
            zorder=1,
        )
        ax.add_patch(rect)

        ax.plot(
            [x - box_width / 2, x + box_width / 2],
            [med, med],
            color=box_col,
            linewidth=median_width,
            alpha=box_line_alpha,
            zorder=2,
        )

        ax.plot(
            [x, x], [q3, w_high],
            color=box_col,
            linewidth=whisker_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x, x], [q1, w_low],
            color=box_col,
            linewidth=whisker_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_high, w_high],
            color=box_col,
            linewidth=cap_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_low, w_low],
            color=box_col,
            linewidth=cap_width,
            alpha=box_line_alpha,
            zorder=1
        )

        if subgroup_col is None:
            jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(vals))
            ax.scatter(
                np.full_like(vals, x, dtype=float) + jitter_x,
                vals,
                s=point_size,
                color=box_col,
                alpha=0.85,
                edgecolors="none",
                zorder=3,
            )
        else:
            g_df = sub.loc[g_mask, [value_col, subgroup_col]]
            for sub_name, sdf in g_df.groupby(subgroup_col):
                v = sdf[value_col].values.astype(float)
                v = v[np.isfinite(v)]
                if len(v) == 0:
                    continue

                jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(v))
                col = subgroup_palette.get(sub_name, "#555555")

                ax.scatter(
                    np.full_like(v, x, dtype=float) + jitter_x,
                    v,
                    s=point_size,
                    color=col,
                    alpha=0.9,
                    edgecolors="none",
                    zorder=3,
                    label=sub_name,
                )

    ax.set_xticks(xs)
    ax.set_xticklabels(group_order, rotation=45, ha="right")
    ax.set_ylim(y_min, y_max)
    ax.margins(x=0.05)

    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if title is not None:
        ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if subgroup_col is not None and show_subgroup_legend:
        handles, labels = ax.get_legend_handles_labels()
        handle_by_label = dict(zip(labels, handles))

        if subgroup_order is not None:
            legend_labels = [s for s in subgroup_order if s in handle_by_label]
        else:
            legend_labels = [s for s in sub_levels if s in handle_by_label]

        legend_handles = [handle_by_label[l] for l in legend_labels]

        ax.legend(
            legend_handles,
            legend_labels,
            title=subgroup_col,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
            fontsize="x-small"
        )

    return fig, ax


In [ ]:
fig, ax = plot_group_box_dots(
    df=df_pc_with_meta,
    value_col="expansion_index",
    group_col="Period_merged",
    group_order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    figsize=(2, 3.5),
    ylabel="Expansion index",
    meta_color_map=meta_color_map,
    box_line_alpha=0.3,
    box_line_width=1.2,

)
plt.savefig('figures/Fig2e_CD8_expansion_box.pdf', dpi=300, bbox_inches="tight")
plt.show


In [ ]:
group_col = "Period_merged"
value_col = "expansion_index"

df = df_pc_with_meta[[group_col, value_col]].dropna().copy()

groups = df[group_col].dropna().drop_duplicates().tolist()
if len(groups) != 2:
    raise ValueError(f"{group_col} 应该正好有两类，当前为 {len(groups)}：{groups}")

g1, g2 = groups
A = df.loc[df[group_col] == g1, value_col].to_numpy(float)
B = df.loc[df[group_col] == g2, value_col].to_numpy(float)

if A.size == 0 or B.size == 0:
    raise ValueError("两组中至少有一组没有有效数据。")

U, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"[Mann–Whitney U] {g1} vs {g2}: U={U:.4g}, p={p:.4g}, n1={A.size}, n2={B.size}")

def q(x, q): return float(np.quantile(x, q)) if len(x) else np.nan
summary = pd.DataFrame({
    "group": [g1, g2],
    "n": [A.size, B.size],
    "median": [np.median(A), np.median(B)],
    "Q1": [q(A, 0.25), q(B, 0.25)],
    "Q3": [q(A, 0.75), q(B, 0.75)],
})
summary["IQR"] = summary["Q3"] - summary["Q1"]
print("\nSummary:")
print(summary.to_string(index=False))


# Fig2g

In [ ]:
def _expansion_for_one_group(
    adata,
    subgroup_col: str,
    sample_col: str,
    target_col: str,
    dropna_clonotype: bool,
    log_base: str,
    neg_epsilon: float,
    min_cells_subgroup=None,
    min_clonotypes_subgroup=None,
    min_cells_pair=None,
    min_clonotypes_pair=None,
):
    if min_cells_pair is None:
        min_cells_pair = min_cells_subgroup
    if min_clonotypes_pair is None:
        min_clonotypes_pair = min_clonotypes_subgroup

    df_sub = scirpy_expansion_index(
        adata,
        groupby=subgroup_col,
        target_col=target_col,
        dropna_clonotype=dropna_clonotype,
        log_base=log_base,
        return_details=False,
        neg_epsilon=neg_epsilon,
        min_cells_per_group=min_cells_subgroup,
        min_clonotypes_per_group=min_clonotypes_subgroup,
    )

    df_pair = scirpy_expansion_index(
        adata,
        groupby=(subgroup_col, sample_col),
        target_col=target_col,
        dropna_clonotype=dropna_clonotype,
        log_base=log_base,
        return_details=False,
        neg_epsilon=neg_epsilon,
        min_cells_per_group=min_cells_pair,
        min_clonotypes_per_group=min_clonotypes_pair,
    )

    return df_sub, df_pair

def plot_expansion_index_bar_dots_mirrored(
    adata,
    *,
    group_col: str = "Period_merged",
    group_levels: tuple = ("Adult", "Child"),
    subgroup_col: str = "anno_TKA",
    sample_col: str = "Sample_ID",
    target_col: str = "clone_id",
    subgroup_order=None,
    dropna_clonotype: bool = True,
    log_base: str = "e",
    neg_epsilon: float = 1e-12,
    min_cells_subgroup=None,
    min_clonotypes_subgroup=None,
    min_cells_pair=None,
    min_clonotypes_pair=None,
    figsize=(12, 5),

    bar_colors=None,
    bar_alpha: float = 0.8,
    bar_width: float = 0.9,

    jitter: float = 0.10,
    point_size: float = 18,
    point_alpha: float = 0.9,

    title: str = "",
    random_state: int = 0,

    disease_col: str = "Disease",
    disease_order=None,
    disease_palette=None,

    meta_color_map=None,

    show_group_legend: bool = True,
    show_disease_legend: bool = True,

    ylim_quantile: float = 0.98,
    manual_ylim=None,

    right_margin: float = 0.78,

    return_data: bool = False,
):
    rng = np.random.default_rng(random_state)
    g_top, g_bottom = group_levels

    def make_auto_palette(levels, used_colors=None):
        base_tab20 = list(cm.get_cmap("tab20").colors)

        def to_morandi(c, mix=0.85, grey=(0.94, 0.94, 0.94)):
            r, g, b = c[:3]
            return (
                r * mix + grey[0] * (1 - mix),
                g * mix + grey[1] * (1 - mix),
                b * mix + grey[2] * (1 - mix),
            )

        auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if used_colors is not None:
            used_colors = set(c.lower() for c in used_colors)
            auto_colors = [c for c in auto_colors if c.lower() not in used_colors]

        if len(auto_colors) == 0:
            auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if len(auto_colors) < len(levels):
            repeats = int(np.ceil(len(levels) / len(auto_colors)))
            auto_colors = (auto_colors * repeats)[:len(levels)]
        else:
            auto_colors = auto_colors[:len(levels)]

        return dict(zip(levels, auto_colors))

    levels_present = adata.obs[group_col].dropna().unique().tolist()
    if g_top not in levels_present or g_bottom not in levels_present:
        raise ValueError(
            f"{group_col} 中找不到指定的两个 level: {group_levels}；实际有: {levels_present}"
        )

    ad_top = adata[adata.obs[group_col] == g_top].copy()
    ad_bottom = adata[adata.obs[group_col] == g_bottom].copy()

    df_sub_top, df_pair_top = _expansion_for_one_group(
        ad_top,
        subgroup_col,
        sample_col,
        target_col,
        dropna_clonotype,
        log_base,
        neg_epsilon,
        min_cells_subgroup=min_cells_subgroup,
        min_clonotypes_subgroup=min_clonotypes_subgroup,
        min_cells_pair=min_cells_pair,
        min_clonotypes_pair=min_clonotypes_pair,
    )

    df_sub_bottom, df_pair_bottom = _expansion_for_one_group(
        ad_bottom,
        subgroup_col,
        sample_col,
        target_col,
        dropna_clonotype,
        log_base,
        neg_epsilon,
        min_cells_subgroup=min_cells_subgroup,
        min_clonotypes_subgroup=min_clonotypes_subgroup,
        min_cells_pair=min_cells_pair,
        min_clonotypes_pair=min_clonotypes_pair,
    )

    if df_sub_top.empty and df_sub_bottom.empty:
        raise ValueError("两个 group 计算 expansion_index 后都是空表，请检查过滤阈值和 clonotype 信息。")

    def _add_disease(df_pair, ad):
        if df_pair.empty:
            df_pair = df_pair.copy()
            df_pair[disease_col] = np.nan
            return df_pair
        meta = (
            ad.obs[[sample_col, disease_col]]
            .dropna(subset=[sample_col])
            .drop_duplicates()
        )
        return df_pair.merge(meta, on=sample_col, how="left")

    df_pair_top = _add_disease(df_pair_top, ad_top)
    df_pair_bottom = _add_disease(df_pair_bottom, ad_bottom)

    if subgroup_order is None:
        s = adata.obs[subgroup_col]
        if hasattr(s, "cat"):
            base_order = list(s.cat.categories)
        else:
            base_order = list(pd.unique(s))
        subgroup_order = [
            g for g in base_order
            if (g in df_sub_top[subgroup_col].values) or (g in df_sub_bottom[subgroup_col].values)
        ]
    else:
        subgroup_order = [
            g for g in subgroup_order
            if (g in df_sub_top[subgroup_col].values) or (g in df_sub_bottom[subgroup_col].values)
        ]

    if len(subgroup_order) == 0:
        raise ValueError("subgroup_order 中的水平，在两个 group 的 df_sub 里都不存在。")

    if bar_colors is None:
        if meta_color_map is not None:
            bar_colors = (
                meta_color_map.get(g_top, "steelblue"),
                meta_color_map.get(g_bottom, "#f6c49b"),
            )
        else:
            bar_colors = ("steelblue", "#f6c49b")

    if disease_order is None:
        dis_levels = list(pd.unique(adata.obs[disease_col].dropna()))
    else:
        dis_levels = [d for d in disease_order if d in adata.obs[disease_col].values]

    if len(dis_levels) == 0:
        dis_levels = ["_NA_"]

    if disease_palette is None:
        disease_palette = {}

        if meta_color_map is not None:
            for d in dis_levels:
                if d in meta_color_map:
                    disease_palette[d] = meta_color_map[d]

        missing_levels = [d for d in dis_levels if d not in disease_palette]
        if len(missing_levels) > 0:
            auto_pal = make_auto_palette(
                missing_levels,
                used_colors=disease_palette.values()
            )
            disease_palette.update(auto_pal)

    df_top_idx = df_sub_top.set_index(subgroup_col)
    df_bot_idx = df_sub_bottom.set_index(subgroup_col)

    bar_top = []
    bar_bottom = []
    for g in subgroup_order:
        bar_top.append(float(df_top_idx.loc[g, "expansion_index"]) if g in df_top_idx.index else 0.0)
        bar_bottom.append(float(df_bot_idx.loc[g, "expansion_index"]) if g in df_bot_idx.index else 0.0)

    bar_top = np.asarray(bar_top, dtype=float)
    bar_bottom = np.asarray(bar_bottom, dtype=float)

    x = np.arange(len(subgroup_order))
    pos_map = {g: i for i, g in enumerate(subgroup_order)}

    fig, ax = plt.subplots(figsize=figsize)

    ax.bar(
        x,
        bar_top,
        width=bar_width,
        color=bar_colors[0],
        edgecolor="none",
        alpha=bar_alpha,
        zorder=1,
    )

    ax.bar(
        x,
        -bar_bottom,
        width=bar_width,
        color=bar_colors[1],
        edgecolor="none",
        alpha=bar_alpha,
        zorder=1,
    )

    for g in subgroup_order:
        g_df = df_pair_top[df_pair_top[subgroup_col] == g]
        if g_df.empty:
            continue
        for dis, sdf in g_df.groupby(disease_col):
            vals = sdf["expansion_index"].values.astype(float)
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                continue
            jit = rng.normal(loc=0.0, scale=jitter, size=len(vals))
            col = disease_palette.get(dis, "#555555")
            ax.scatter(
                np.full_like(vals, pos_map[g], dtype=float) + jit,
                vals,
                s=point_size,
                color=col,
                alpha=point_alpha,
                edgecolors="none",
                zorder=2,
            )

    for g in subgroup_order:
        g_df = df_pair_bottom[df_pair_bottom[subgroup_col] == g]
        if g_df.empty:
            continue
        for dis, sdf in g_df.groupby(disease_col):
            vals = sdf["expansion_index"].values.astype(float)
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                continue
            jit = rng.normal(loc=0.0, scale=jitter, size=len(vals))
            col = disease_palette.get(dis, "#555555")
            ax.scatter(
                np.full_like(vals, pos_map[g], dtype=float) + jit,
                -vals,
                s=point_size,
                color=col,
                alpha=point_alpha,
                edgecolors="none",
                zorder=2,
            )

    all_vals = np.concatenate([
        bar_top,
        bar_bottom,
        df_pair_top["expansion_index"].to_numpy(dtype=float) if len(df_pair_top) > 0 else np.array([]),
        df_pair_bottom["expansion_index"].to_numpy(dtype=float) if len(df_pair_bottom) > 0 else np.array([]),
    ])
    all_vals = all_vals[np.isfinite(all_vals)]

    if manual_ylim is not None:
        ax.set_ylim(manual_ylim)
    else:
        if all_vals.size == 0:
            ymax = 1.0
        else:
            ymax = np.quantile(np.abs(all_vals), ylim_quantile)
            ymax = max(ymax, 1e-6) * 1.15
        ax.set_ylim(-ymax, ymax)

    ax.axhline(0, color="black", linewidth=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(subgroup_order, rotation=45, ha="right")
    ax.set_ylabel("Expansion index")
    ax.set_xlabel(subgroup_col)
    ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if show_group_legend:
        group_handles = [
            Patch(facecolor=bar_colors[0], edgecolor="none", alpha=bar_alpha, label=str(g_top)),
            Patch(facecolor=bar_colors[1], edgecolor="none", alpha=bar_alpha, label=str(g_bottom)),
        ]
        group_legend = ax.legend(
            handles=group_handles,
            title=group_col,
            bbox_to_anchor=(1.02, 1.00),
            loc="upper left",
            frameon=False,
            fontsize="x-small",
            title_fontsize="small"
        )
        ax.add_artist(group_legend)

    if show_disease_legend:
        legend_labels = [d for d in dis_levels if d in disease_palette]
        disease_handles = [
            Line2D(
                [0], [0],
                marker="o",
                linestyle="",
                color=disease_palette[d],
                label=d,
                markersize=5,
            )
            for d in legend_labels
        ]

        disease_anchor_y = 0.62 if show_group_legend else 1.00

        ax.legend(
            disease_handles,
            legend_labels,
            title=disease_col,
            bbox_to_anchor=(1.02, disease_anchor_y),
            loc="upper left",
            frameon=False,
            fontsize="x-small",
            title_fontsize="small"
        )

    fig.subplots_adjust(right=right_margin)

    if return_data:
        return fig, ax, {
            "top_sub": df_sub_top,
            "top_pair": df_pair_top,
            "bottom_sub": df_sub_bottom,
            "bottom_pair": df_pair_bottom,
            "disease_palette": disease_palette,
            "bar_colors": bar_colors,
        }
    else:
        return fig, ax


In [ ]:
fig, ax, res = plot_expansion_index_bar_dots_mirrored(
    CD8_T_clone_tumor,
    group_col="Period_merged",
    group_levels=("Child", "Adult"),
    subgroup_col="anno_CD8T",
    sample_col="Sample_ID",
    target_col="clone_Sample",
    disease_col="Disease",
    disease_order=disease_order,
    figsize=(10, 6),

    min_cells_subgroup=100,
    min_clonotypes_subgroup=5,

    min_cells_pair=20,
    min_clonotypes_pair=3,

    return_data=True,
    meta_color_map=meta_color_map,
    bar_alpha=0.3,
    bar_width=0.8,
    ylim_quantile=0.97,

)
plt.savefig('figures/Fig2g_CD8_expansion_by_state.pdf', dpi=300, bbox_inches="tight")
plt.show()


# Fig2i

In [ ]:
def make_clone_summary_table(
    adata: AnnData,
    *,
    clone_col: str = "clone_Sample",
    sample_col: str = "Sample_ID",
    disease_col: str = "Disease",
    period_col: str = "Period_merged",
    cluster_col: str = "anno_TKA",

    large_thresh: int = 20,
    connected_thresh: int = 3,
    min_connected_for_frequent: int = 3,

    filter_large: bool = True,
    filter_frequent: bool = True,
    top_n_per_sample: int = None,
) -> pd.DataFrame:

    obs = adata.obs[[clone_col, sample_col, disease_col, period_col, cluster_col]].copy()

    obs = obs[~obs[clone_col].isna()].copy()

    clone_summary = (
        obs
        .groupby(clone_col)
        .agg(
            Sample_ID     = (sample_col, "first"),
            Disease       = (disease_col, "first"),
            Period_merged = (period_col, "first"),
            cell_number   = (clone_col, "size"),
        )
    )

    anno_counts = (
        obs
        .groupby([clone_col, cluster_col])
        .size()
        .unstack(fill_value=0)
    )

    cluster_cols = list(anno_counts.columns)

    df_clone = (
        clone_summary
        .join(anno_counts, how="left")
        .reset_index()
        .rename(columns={clone_col: "clone_id"})
    )

    for col in cluster_cols:
        frac_col = f"{col}_frac"

        df_clone[frac_col] = np.where(
            df_clone["cell_number"] > 0,
            df_clone[col] / df_clone["cell_number"],
            np.nan,
        )

    df_clone["large_clone"] = df_clone["cell_number"] >= large_thresh

    if filter_large:
        df_clone = df_clone[df_clone["large_clone"]].copy()

    connected_cols = []
    for col in cluster_cols:
        new_col = f"{col}_connected"
        df_clone[new_col] = df_clone[col] >= connected_thresh
        connected_cols.append(new_col)

    df_clone["frequent_share"] = df_clone[connected_cols].sum(axis=1) >= min_connected_for_frequent

    if filter_frequent:
        df_clone = df_clone[df_clone["frequent_share"]].copy()

    if top_n_per_sample is not None:
        df_clone = (
            df_clone.sort_values(["Sample_ID", "cell_number"], ascending=[True, False])
                    .groupby("Sample_ID", as_index=False, group_keys=False)
                    .head(top_n_per_sample)
                    .reset_index(drop=True)
        )

    return df_clone


# Fig2i

In [ ]:
df_clone = make_clone_summary_table(
    CD8_T_clone_tumor,
    clone_col="clone_Sample",
    sample_col="Sample_ID",
    disease_col="Disease",
    period_col="Period_merged",
    cluster_col="anno_CD8T",
    large_thresh=5,
    connected_thresh=1,
    min_connected_for_frequent=3,
    filter_large=True,
    filter_frequent=True,
)


In [ ]:
df_clone_filterd= (
    df_clone
    .groupby("Disease", group_keys=False)
    .apply(lambda g: g.nlargest(5, "cell_number"))
    .reset_index(drop=True)
)


In [ ]:
def make_df_path(
    df_clone: pd.DataFrame,
    *,
    states: List[str],
    stage_map: Dict[str, int],
    clone_id_col: str = "clone_id",
    group_col: Optional[str] = "Period_merged",
    connected_suffix: str = "_connected",
    frac_suffix: str = "_frac",
    prefer: str = "max_frac",
    weight_col: Optional[str] = None,
    include_none: bool = True,
    none_label: str = "None",
    min_connected: int = 1,
) -> Tuple[pd.DataFrame, Dict[int, List[str]]]:

    if clone_id_col not in df_clone.columns:
        raise ValueError(f"df_clone 缺少列 {clone_id_col}")

    missing_stage = [s for s in states if s not in stage_map]
    if missing_stage:
        raise ValueError(f"stage_map 缺少这些 state：{missing_stage}")

    stage_ids = sorted(set(stage_map[s] for s in states))
    if stage_ids != list(range(1, max(stage_ids) + 1)):
        raise ValueError(f"stage_map 的 stage 编号需要从1开始连续(1..K)，当前：{stage_ids}")
    K = max(stage_ids)

    conn_cols = [f"{s}{connected_suffix}" for s in states]
    missing = [c for c in conn_cols if c not in df_clone.columns]
    if missing:
        raise ValueError(f"df_clone 缺少这些 connected 列：{missing}")

    if prefer == "max_frac":
        frac_cols = [f"{s}{frac_suffix}" for s in states]

    orders: Dict[int, List[str]] = {k: [s for s in states if stage_map[s] == k] for k in range(1, K + 1)}

    if include_none:
        for k in range(1, K + 1):
            if none_label not in orders[k]:
                orders[k] = orders[k] + [none_label]

    def pick_state(row: pd.Series, stage_states: List[str]) -> str:

        connected = []
        for s in stage_states:
            if s == none_label:
                continue
            v = row.get(f"{s}{connected_suffix}", False)

            ok = (bool(v) if isinstance(v, (bool, np.bool_)) else (pd.to_numeric(v, errors="coerce") >= min_connected))
            if ok:
                connected.append(s)

        if not connected:
            return none_label
        if len(connected) == 1:
            return connected[0]

        if prefer == "max_frac":

            def score(s: str):
                frac = row.get(f"{s}{frac_suffix}", np.nan)
                frac_val = float(frac) if pd.notna(frac) else -1e18
                return (-frac_val, stage_states.index(s))
            return sorted(connected, key=score)[0]

        return connected[0]

    df_path = pd.DataFrame({clone_id_col: df_clone[clone_id_col].values})

    if group_col is not None:
        if group_col not in df_clone.columns:
            raise ValueError(f"group_col='{group_col}' 不在 df_clone.columns 中")
        df_path[group_col] = df_clone[group_col].fillna("Unknown").astype(str).values

    for k in range(1, K + 1):
        st = [s for s in orders[k] if s != none_label]

        df_path[f"stage{k}"] = df_clone.apply(lambda r: pick_state(r, st), axis=1)

    if weight_col is None:
        df_path["weight"] = 1.0
    else:
        if weight_col not in df_clone.columns:
            raise ValueError(f"weight_col='{weight_col}' 不在 df_clone.columns 中")
        df_path["weight"] = pd.to_numeric(df_clone[weight_col], errors="coerce").fillna(0.0).astype(float).values

    return df_path, orders


In [ ]:
def _ribbon(ax, x0, x1, y0_lo, y0_hi, y1_lo, y1_hi, color, alpha=0.6):
    dx = (x1 - x0) * 0.5
    verts = [
        (x0, y0_lo),
        (x0 + dx, y0_lo),
        (x1 - dx, y1_lo),
        (x1, y1_lo),
        (x1, y1_hi),
        (x1 - dx, y1_hi),
        (x0 + dx, y0_hi),
        (x0, y0_hi),
        (x0, y0_lo),
    ]
    codes = [
        Path.MOVETO,
        Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.LINETO,
        Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.CLOSEPOLY,
    ]
    ax.add_patch(PathPatch(Path(verts, codes), facecolor=color, edgecolor="none", alpha=alpha))

def clone_ribbon_plot_grouped(
    df: pd.DataFrame,
    *,
    stage_cols=("stage1", "stage2", "stage3", "stage4"),
    group_col="Period_merged",
    group_order=("Adult", "Child"),
    group_colors=None,
    weight_col="weight",
    max_segments=25,
    max_ribbons_per_group=400,
    barwidth=0.38,

    bar_alpha=1.0,
    ribbon_alpha=0.55,

    segment_colors=None,
    other_label="other",
    na_label="None",
    other_color="#D9D9D9",
    figsize=(12, 4),
    show_period_legend=True,
    show_state_legend=True,
    legend_anchor=(1.02, 1.0),
    legend_loc="upper left",
    ax=None,

    bar_edgecolor="none",
    bar_linewidth=0.0,
):
    df = df.copy()

    for c in stage_cols:
        if c not in df.columns:
            raise ValueError(f"missing column: {c}")
    if group_col not in df.columns:
        raise ValueError(f"missing column: {group_col}")

    if weight_col is None:
        df["_w"] = 1.0
    else:
        if weight_col not in df.columns:
            raise ValueError(f"missing weight_col: {weight_col}")
        df["_w"] = pd.to_numeric(df[weight_col], errors="coerce").fillna(0.0).astype(float)

    df[group_col] = df[group_col].where(df[group_col].notna(), "Unknown")
    for c in stage_cols:
        df[c] = df[c].where(df[c].notna(), na_label).astype(str)

    present_groups = [g for g in group_order if g in set(df[group_col])]
    if not present_groups:
        present_groups = list(pd.unique(df[group_col]))

    if group_colors is None:
        group_colors = {"Adult": "#F4A6A6", "Child": "#35C9CF", "Unknown": "#BDBDBD"}
    group_colors = {g: group_colors.get(g, "#BDBDBD") for g in present_groups}

    if segment_colors is None:
        segment_colors = {}
    segment_colors = dict(segment_colors)
    segment_colors.setdefault(na_label, "#BDBDBD")

    ncols = len(stage_cols)
    x = np.arange(1, ncols + 1, dtype=float)
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    tops = {c: {} for c in stage_cols}
    states_seen = set()

    for j, c in enumerate(stage_cols):
        gw = (
            df.groupby([c, group_col], observed=True)["_w"]
            .sum()
            .unstack(fill_value=0.0)
        )

        total = gw.sum(axis=1).sort_values(ascending=False)
        kept = list(total.index[:max_segments]) if max_segments is not None else list(total.index)
        other_states = [s for s in total.index if s not in kept]

        bottom = 0.0

        if other_states:
            byg = gw.loc[other_states].sum(axis=0)
            h_total = float(byg.sum())
            if h_total > 0:
                ax.bar(
                    x[j],
                    h_total,
                    width=barwidth,
                    bottom=bottom,
                    color=other_color,
                    edgecolor=bar_edgecolor,
                    linewidth=bar_linewidth,
                    alpha=bar_alpha,
                )
            tops[c][other_label] = {}
            cur = bottom
            for g in present_groups:
                cur += float(byg.get(g, 0.0))
                tops[c][other_label][g] = cur
            bottom += h_total
            states_seen.add(other_label)

        for state in kept[::-1]:
            byg = gw.loc[state] if state in gw.index else pd.Series(0.0, index=present_groups)
            h_total = float(byg.sum())
            face = segment_colors.get(state, other_color)

            if h_total > 0:
                ax.bar(
                    x[j],
                    h_total,
                    width=barwidth,
                    bottom=bottom,
                    color=face,
                    edgecolor=bar_edgecolor,
                    linewidth=bar_linewidth,
                    alpha=bar_alpha,
                )

            tops[c][state] = {}
            cur = bottom
            for g in present_groups:
                cur += float(byg.get(g, 0.0))
                tops[c][state][g] = cur

            bottom += h_total
            states_seen.add(state)

    x_left = x + barwidth / 2
    x_right = x - barwidth / 2

    for g in present_groups:
        df_g = df[df[group_col] == g]
        if df_g.empty:
            continue

        combos = (
            df_g.groupby(list(stage_cols), observed=True)["_w"]
            .sum()
            .reset_index()
            .sort_values(
                by=["_w"] + list(stage_cols),
                ascending=[False] + [True] * len(stage_cols)
            )
        )

        tmp = deepcopy(tops)

        for _, row in combos.iloc[:max_ribbons_per_group].iterrows():
            h = float(row["_w"])
            if h <= 0:
                continue

            breaks = []
            for c in stage_cols:
                s = row[c] if row[c] is not None else na_label
                if s not in tmp[c]:
                    s = other_label
                top = float(tmp[c][s].get(g, 0.0))
                breaks.append((top - h, top))
                tmp[c][s][g] = top - h

            for k in range(ncols - 1):
                _ribbon(
                    ax,
                    x_left[k], x_right[k + 1],
                    breaks[k][0], breaks[k][1],
                    breaks[k + 1][0], breaks[k + 1][1],
                    color=group_colors.get(g, "#BDBDBD"),
                    alpha=ribbon_alpha,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(stage_cols)
    ax.set_yticks([])
    ax.set_ylabel("")
    ax.tick_params(axis="y", left=False, labelleft=False)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.grid(False)
    ax.set_xlim(0.5, ncols + 0.5)

    handles, labels = [], []

    if show_period_legend:
        for g in present_groups:
            handles.append(
                Line2D([0], [0], color=group_colors[g], lw=10, alpha=ribbon_alpha)
            )
            labels.append(str(g))

    if show_state_legend:
        state_list = [s for s in states_seen if s not in (other_label,)]
        ordered = [s for s in segment_colors.keys() if s in state_list]
        ordered += [s for s in sorted(state_list) if s not in ordered]

        for s in ordered:
            handles.append(
                Patch(
                    facecolor=segment_colors.get(s, other_color),
                    edgecolor="none" if bar_edgecolor == "none" else bar_edgecolor,
                    linewidth=bar_linewidth,
                    alpha=bar_alpha,
                )
            )
            labels.append(str(s))

    if handles:
        ax.legend(handles, labels, frameon=False, loc=legend_loc, bbox_to_anchor=legend_anchor)

    return ax


In [ ]:
states = ["CD8_TN_c2","CD8_TN_c9",
              'CD8_t-TEM_c8',
              "CD8_TEM_c1","CD8_TEM_c3","CD8_TEM_c5","CD8_TEM_c6",
              'CD8_TRM_c4',"CD8_TRM_c11",
              'CD8_TEMRA_c7','CD8_TNK_c10','CD8_TEX_c12',
              'CD8_TISG_c13']
stage = {
  "CD8_TN_c2": 1,
   "CD8_TN_c9": 1,
    "CD8_t-TEM_c8": 1,
  "CD8_TEM_c1": 2,
  "CD8_TEM_c3":1,
  "CD8_TEM_c5": 2,
  "CD8_TEM_c6":2,
  "CD8_TRM_c4": 2,
  'CD8_TISG_c13':2,
  "CD8_TRM_c11": 3,
  "CD8_TEMRA_c7": 3,
   'CD8_TEX_c12':3,
    'CD8_TNK_c10':3
}


In [ ]:
df_path, orders = make_df_path(
    df_clone_filterd,
    states=states,
    stage_map=stage,
    clone_id_col="clone_id",
    group_col="Period_merged",
    prefer="max_frac",
    weight_col=None
)


In [ ]:
ax = clone_ribbon_plot_grouped(
    df_path,
    stage_cols=("stage1","stage2","stage3"),
    group_col="Period_merged",
    group_order=("Adult","Child"),
    group_colors=meta_color_map,
    barwidth=0.20,
    weight_col="weight",
    segment_colors=cd8_color_map,
    max_segments=25,
    max_ribbons_per_group=400,
    figsize=(12,5),
    show_period_legend=True,
    show_state_legend=True,
    bar_alpha=0.85,
    ribbon_alpha=0.4,
    bar_edgecolor="white",
    bar_linewidth=0.0
)
plt.tight_layout()
plt.savefig('figures/Fig2i_CD8_clone_stages.pdf', dpi=300, bbox_inches="tight")
plt.show()


# fig2f

In [ ]:
def _morandi_cmap(base_cmap: Union[str, mpl.colors.Colormap],
                  mix: float = 0.8,
                  grey=(0.88, 0.86, 0.84)) -> mpl.colors.Colormap:
    if isinstance(base_cmap, str):
        base = mpl.cm.get_cmap(base_cmap)(np.linspace(0, 1, 256))
    else:
        base = base_cmap(np.linspace(0, 1, 256))
    rgb = base[:, :3]
    grey = np.array(grey)[None, :3]
    rgb_new = rgb * mix + grey * (1 - mix)
    base[:, :3] = rgb_new
    return mpl.colors.ListedColormap(base)

def plot_repertoire_overlap_group(
    adata,
    groupby: str = "anno_TKA",
    *,
    group_col: str = "Period_merged",
    group_levels: Optional[Sequence[str]] = None,
    target_col: str = "clone_Sample",
    overlap_measure: str = "jaccard",
    overlap_threshold: float = None,
    fraction: Optional[bool] = True,
    cmap_lower: Union[str, mpl.colors.Colormap] = "Blues",
    cmap_upper: Union[str, mpl.colors.Colormap] = "Reds",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    figsize: Tuple[float, float] = (6, 6),
    reorder_labels: Optional[Sequence[str]] = None,
):

    if isinstance(fraction, str) or fraction is None:
        frac_arg = fraction
    elif isinstance(fraction, bool):
        frac_arg = groupby if fraction else None
    else:
        frac_arg = None

    if group_levels is None:
        levels = adata.obs[group_col].dropna().unique().tolist()
        if len(levels) < 2:
            raise ValueError(f"{group_col} 里不到两个 level，不能做双组比较。")
        if len(levels) > 2:
            levels = levels[:2]
        group_levels = levels
    g1, g2 = group_levels

    adata1 = adata[adata.obs[group_col] == g1].copy()
    adata2 = adata[adata.obs[group_col] == g2].copy()

    df1, dist1, lk1 = ir.tl.repertoire_overlap(
        adata1,
        groupby=groupby,
        target_col=target_col,
        overlap_measure=overlap_measure,
        overlap_threshold=overlap_threshold,
        fraction=frac_arg,
        inplace=False,
    )
    df2, dist2, lk2 = ir.tl.repertoire_overlap(
        adata2,
        groupby=groupby,
        target_col=target_col,
        overlap_measure=overlap_measure,
        overlap_threshold=overlap_threshold,
        fraction=frac_arg,
        inplace=False,
    )

    mat1 = 1.0 - sc_distance.squareform(dist1)
    mat2 = 1.0 - sc_distance.squareform(dist2)

    labels1 = list(df1.index)
    labels2 = list(df2.index)

    if reorder_labels is not None:
        labels_all = [lab for lab in reorder_labels if (lab in labels1) or (lab in labels2)]
    else:
        if groupby in adata.obs.columns:
            s = adata.obs[groupby]
            if hasattr(s, "cat"):
                base_order = list(s.cat.categories)
            else:
                base_order = list(pd.unique(s))
            labels_all = [lab for lab in base_order if (lab in labels1) or (lab in labels2)]
        else:
            labels_all = sorted(set(labels1) | set(labels2))

    n = len(labels_all)
    if n == 0:
        raise ValueError("两个分组里在 groupby 上没有任何水平。")

    idx1 = {lab: i for i, lab in enumerate(labels1)}
    idx2 = {lab: i for i, lab in enumerate(labels2)}

    sim1_full = np.full((n, n), np.nan, dtype=float)
    sim2_full = np.full((n, n), np.nan, dtype=float)
    for i, li in enumerate(labels_all):
        for j, lj in enumerate(labels_all):
            if li in idx1 and lj in idx1:
                sim1_full[i, j] = mat1[idx1[li], idx1[lj]]
            if li in idx2 and lj in idx2:
                sim2_full[i, j] = mat2[idx2[li], idx2[lj]]

    mask_lower_off = np.triu(np.ones((n, n), dtype=bool), k=0)
    sim1_plot = np.ma.masked_where(mask_lower_off, sim1_full)

    mask_upper_off = np.tril(np.ones((n, n), dtype=bool), k=0)
    sim2_plot = np.ma.masked_where(mask_upper_off, sim2_full)

    if vmin is None or vmax is None:
        valid_vals = np.concatenate([
            sim1_full[~mask_lower_off & np.isfinite(sim1_full)],
            sim2_full[~mask_upper_off & np.isfinite(sim2_full)],
        ])
        if valid_vals.size == 0:
            valid_vals = np.array([0.0, 1.0])
        if vmin is None:
            vmin = float(np.nanmin(valid_vals))
        if vmax is None:
            vmax = float(np.nanmax(valid_vals))
        if vmin == vmax:
            vmin -= 1e-6
            vmax += 1e-6

    cmap_lower_m = _morandi_cmap(cmap_lower)
    cmap_upper_m = _morandi_cmap(cmap_upper)

    fig, ax = plt.subplots(figsize=figsize)

    im1 = ax.imshow(sim1_plot, cmap=cmap_lower_m, vmin=vmin, vmax=vmax)
    im2 = ax.imshow(sim2_plot, cmap=cmap_upper_m, vmin=vmin, vmax=vmax)

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels_all, rotation=90, ha="right", fontsize=8)
    ax.set_yticklabels(labels_all, fontsize=8)

    ax.set_xlabel(groupby, fontsize=10)
    ax.set_ylabel(groupby, fontsize=10)

    for spine in ax.spines.values():
        spine.set_visible(False)

    cax_right_top = fig.add_axes([0.93, 0.62, 0.01, 0.22])
    cax_right_bottom = fig.add_axes([0.93, 0.28, 0.01, 0.22])

    cbar2 = fig.colorbar(im2, cax=cax_right_top)
    cbar2.ax.tick_params(labelsize=8)
    cbar2.set_label(f"{g2} (1 - {overlap_measure})", fontsize=9)

    cbar1 = fig.colorbar(im1, cax=cax_right_bottom)
    cbar1.ax.tick_params(labelsize=8)
    cbar1.set_label(f"{g1} (1 - {overlap_measure})", fontsize=9)

    plt.tight_layout(rect=(0.08, 0.05, 0.90, 0.95))
    return fig, ax


In [ ]:
fig, ax = plot_repertoire_overlap_group(
    CD8_T_clone_tumor,
    groupby="anno_CD8T",
    group_col="Period_merged",
    group_levels=("Adult", "Child"),
    target_col="clone_Sample",
    overlap_measure="jaccard",
    fraction=True,
    cmap_lower="Blues",
    cmap_upper="Reds",
    figsize=(7, 7),
)
plt.savefig('figures/Fig2_clonotype_overlap.pdf', dpi=300, bbox_inches="tight")
plt.show()
